# Data Prep

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

data_dir = Path(".")

support_file = data_dir / "SupportData.xlsx"
gl_account_file = data_dir / "GLAccountData.xlsx"
gl_entry_file = data_dir / "GLEntryData.xlsx"

random_state = 42

Next step will be setting up cleaning functions for standardising and creating more in depth date features (day, month, year etc)

In [2]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(".", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace(r"_+", "_", regex=True)
    )
    return df


def standardise_dates(df):
    df = df.copy()
    date_cols = [col for col in df.columns if "date" in col.lower()]
    
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    
    return df


def make_period_features(df, date_col="Date", prefix=""):
    df = df.copy()
    
    if date_col in df.columns:
        stem = f"{prefix}_" if prefix else ""
        
        df[f"{stem}year"] = df[date_col].dt.year
        df[f"{stem}month"] = df[date_col].dt.month
        df[f"{stem}quarter"] = df[date_col].dt.quarter
        df[f"{stem}period_month"] = df[date_col].dt.to_period("M").astype(str)
        df[f"{stem}period_quarter"] = df[date_col].dt.to_period("Q").astype(str)
    
    return df


def missingness_table(df):
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": df.isna().mean() * 100,
        "data_type": df.dtypes.astype(str)
    })
    
    return result.sort_values("missing_percentage", ascending=False)


def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def normalise_key(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.upper()
    )


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def add_row_id(df, name):
    df = df.copy()
    df[name] = range(1, len(df) + 1)
    return df

def stable_line_id(df, id_cols, prefix="CL"):
    available_cols = [col for col in id_cols if col in df.columns]
    if not available_cols:
        base = pd.Series(range(1, len(df) + 1), index=df.index).astype("string")
    else:
        base = (
            df[available_cols]
            .astype("string")
            .fillna("<missing>")
            .agg("|".join, axis=1)
        )

    hashed = pd.util.hash_pandas_object(base, index=False).astype("uint64")
    return hashed.map(lambda value: f"{prefix}-{int(value):016X}")


def first_existing_cols(df, candidates):
    return [col for col in candidates if col in df.columns]


In [3]:
def normalise_gl_description(series):
    return (
        series.astype("string")
        .str.upper()
        .str.replace(r"\bCLAIMBACKS?\b", "", regex=True)
        .str.replace(r"\bE\d+[A-Z]*\b", "", regex=True)
        .str.replace(r"\bLIMITED\b|\bLTD\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def prepare_gl_account_data(path):
    gl_accounts = clean_columns(pd.read_excel(path, dtype=str))
    gl_accounts["gl_account_code"] = normalise_key(gl_accounts["Code"])
    gl_accounts["gl_account_description"] = gl_accounts.get("Description", pd.Series(pd.NA, index=gl_accounts.index))
    gl_accounts["gl_supplier_key"] = normalise_gl_description(gl_accounts["gl_account_description"])
    gl_accounts["gl_current_balance"] = pd.to_numeric(gl_accounts.get("CurrentBalance", pd.NA), errors="coerce")
    gl_accounts["gl_current_balance_abs"] = gl_accounts["gl_current_balance"].abs()
    return gl_accounts


def prepare_gl_entry_data(path):
    gl_entries = clean_columns(pd.read_excel(path, dtype=str))
    gl_entries["gl_account_code"] = normalise_key(gl_entries["GL_Account_Code"])
    gl_entries["gl_account_description"] = gl_entries.get("Description", pd.Series(pd.NA, index=gl_entries.index))
    gl_entries["gl_supplier_key"] = normalise_gl_description(gl_entries["gl_account_description"])
    gl_entries["gl_total_balance"] = pd.to_numeric(gl_entries.get("Total_Balance", pd.NA), errors="coerce")
    gl_entries["gl_credit_amount"] = pd.to_numeric(gl_entries.get("Credit_Amount", pd.NA), errors="coerce")
    gl_entries["gl_running_balance"] = pd.to_numeric(gl_entries.get("Running_Balance", pd.NA), errors="coerce")
    gl_entries["gl_entry_line_date"] = pd.to_datetime(gl_entries.get("Entry_Line_Date", pd.NA), errors="coerce", dayfirst=True)
    gl_entries["gl_entry_period_month"] = gl_entries["gl_entry_line_date"].dt.to_period("M").astype("string")
    return gl_entries


removing duplicates for raw data files

In [4]:
support_raw = pd.read_excel(support_file, dtype=str)
support_raw = clean_columns(support_raw)
support_raw = standardise_dates(support_raw)

customer_raw = support_raw.copy()

supplier_support_cols = [
    "Date",
    "Contract_Number",
    "Contract_D_ContractNumber",
    "Product_ManufacturerProductCode",
    "Product_Category",
    "Contract_Description",
    "Contract_Expression",
    "Contract_ValidFrom",
    "Contract_ValidTo",
    "Contract_Supplier_Code",
    "Contract_Supplier_Name",
    "UnitClaimAmount",
    "TotalClaimAmount",
    "SalesDeliveryNoteLine_Quantity",
    "SalesInvoiceLine_Quantity",
    "Product_CSQL_InvoiceCostForbranch",
    "Product_CSQL_ListPriceForBranch",
    "Product_CALC_FixedCostForBranch"
]

supplier_raw = (
    support_raw[[col for col in supplier_support_cols if col in support_raw.columns]]
    .drop_duplicates()
    .reset_index(drop=True)
)

customer_raw = add_row_id(customer_raw, "customer_iq_row_id")
supplier_raw = add_row_id(supplier_raw, "supplier_iq_row_id")

print("Support source shape:", support_raw.shape)
print("Customer claim view shape:", customer_raw.shape)
print("Supplier support view shape:", supplier_raw.shape)

display(customer_raw.head())
display(supplier_raw.head())


Support source shape: (73320, 41)
Customer claim view shape: (73320, 42)
Supplier support view shape: (70559, 19)


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,customer_iq_row_id
0,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,...,6.235348,14.21,4.61415752,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1
1,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,...,8.130964,18.53,6.01691336,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2
2,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,...,5.432344,12.38,4.01993456,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3
3,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,...,11.351756,25.87,8.40029944,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4
4,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,...,4.0589,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,5


,Date,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,Product_Category,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,supplier_iq_row_id
0,2024-02-26,0086,039-C-0692,38300,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.366861,8.2012,6,0,6.235348,14.21,4.61415752,1
1,2024-02-26,0086,039-C-0692,38322,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.783139,3.5663,2,0,8.130964,18.53,6.01691336,2
2,2024-02-26,0086,039-C-0692,38204,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.190942,2.3819,2,0,5.432344,12.38,4.01993456,3
3,2024-02-26,0086,039-C-0692,38492,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2.488979,9.9559,4,0,11.351756,25.87,8.40029944,4
4,2024-02-26,0086,039-C-0692,38490,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,0.890474,5.3428,6,0,4.0589,9.25,3.003586,5


GL account summary

In [5]:
gl_accounts = prepare_gl_account_data(gl_account_file)
gl_entries = prepare_gl_entry_data(gl_entry_file)

gl_entry_account_summary = (
    gl_entries
    .sort_values(["gl_account_code", "gl_entry_line_date"])
    .groupby(["gl_account_code", "gl_account_description", "gl_supplier_key"], dropna=False)
    .agg(
        gl_entry_rows=("gl_account_code", "size"),
        gl_first_entry_date=("gl_entry_line_date", "min"),
        gl_last_entry_date=("gl_entry_line_date", "max"),
        gl_total_credit_amount=("gl_credit_amount", "sum"),
        gl_latest_running_balance=("gl_running_balance", "last"),
    )
    .reset_index()
)

gl_account_summary = gl_accounts.merge(
    gl_entry_account_summary,
    on=["gl_account_code", "gl_account_description", "gl_supplier_key"],
    how="left"
)

gl_account_summary["gl_entry_rows"] = gl_account_summary["gl_entry_rows"].fillna(0).astype(int)

print("GL accounts prepared:", gl_accounts.shape)
print("GL entries prepared:", gl_entries.shape)
print("GL account summary:", gl_account_summary.shape)

display(gl_account_summary.head(10))


GL accounts prepared: (21, 9)
GL entries prepared: (170, 14)
GL account summary: (21, 14)


,Code,Description,D_MerlinAccountCode,CurrentBalance,gl_account_code,gl_account_description,gl_supplier_key,gl_current_balance,gl_current_balance_abs,gl_entry_rows,gl_first_entry_date,gl_last_entry_date,gl_total_credit_amount,gl_latest_running_balance
0,01-85001,Supplier13 E071 Claimbacks,NaN,-242784.06,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,-242784.06,242784.06,105,2026-01-05,2026-03-07,242784.06,149850.93
1,01-85002,Supplier2 E005 Claimbacks,NaN,-6240,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,-6240.00,6240.00,1,NaT,NaT,6240.00,6240.00
2,01-85003,Supplier2GB E005GB Claimbacks,NaN,0,01-85003,Supplier2GB E005GB Claimbacks,SUPPLIER2GB,0.00,0.00,0,NaT,NaT,NaN,NaN
3,01-85007,Supplier9 E022 Claimbacks,NaN,-5774.89,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,-5774.89,5774.89,5,2026-04-06,2026-10-07,5774.89,4829.89
4,01-85017,Supplier3 E055 Claimbacks,NaN,-5550,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,-5550.00,5550.00,4,2026-10-04,2026-10-04,5550.00,5550.00
5,01-85018,Supplier5 E059 Claimbacks,NaN,0,01-85018,Supplier5 E059 Claimbacks,SUPPLIER5,0.00,0.00,0,NaT,NaT,NaN,NaN
6,01-85021,Supplier19 E070 Claimbacks,NaN,-9124.28,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,-9124.28,9124.28,4,NaT,NaT,9124.28,9124.28
7,01-85036,Supplier8 E175 Claimbacks,NaN,-4782.68,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,-4782.68,4782.68,2,2026-04-06,2026-04-06,4782.68,2323.37
8,01-85046,Supplier10 E234 Claimbacks,NaN,-925,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,-925.00,925.00,8,2026-08-04,2026-08-04,925.00,925.00
9,01-85056,Supplier17 E269A Claimbacks,NaN,-576,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,-576.00,576.00,1,NaT,NaT,576.00,576.00


In [6]:
print("Customer columns:")
display(pd.Series(customer_raw.columns))

print("Supplier columns:")
display(pd.Series(supplier_raw.columns))


Customer columns:


0                                            Date
1                                           Month
2                                            Year
3                        SalesDeliveryNote_Branch
4                                        Customer
5                                   Customer_Name
6                                         Product
7                 Product_ManufacturerProductCode
8                                 Contract_Number
9                           SourceTransactionType
10                      Contract_D_ContractNumber
11                            SalesInvoice_Number
12                         SalesCreditNote_Number
13                       SalesDeliveryNote_Number
14                         SalesReturnNote_Number
15                              SalesOrder_Number
16                                         Status
17                                UnitClaimAmount
18                               TotalClaimAmount
19                 SalesDeliveryNoteLine_Quantity


Supplier columns:


0                                  Date
1                       Contract_Number
2             Contract_D_ContractNumber
3       Product_ManufacturerProductCode
4                      Product_Category
5                  Contract_Description
6                   Contract_Expression
7                    Contract_ValidFrom
8                      Contract_ValidTo
9                Contract_Supplier_Code
10               Contract_Supplier_Name
11                      UnitClaimAmount
12                     TotalClaimAmount
13       SalesDeliveryNoteLine_Quantity
14            SalesInvoiceLine_Quantity
15    Product_CSQL_InvoiceCostForbranch
16      Product_CSQL_ListPriceForBranch
17      Product_CALC_FixedCostForBranch
18                   supplier_iq_row_id
dtype: object

As stated the process is carried out manually through excel. The main excel sheet is in the form of a monthly view with owed/paid format per each supplier (in rows) so the aim is to translate that through Python/SQL

In [7]:
customer = make_period_features(customer_raw, date_col="Date", prefix="customer")
supplier = make_period_features(supplier_raw, date_col="Date", prefix="supplier")

display(customer.head())
display(supplier.head())


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,customer_iq_row_id,customer_year,customer_month,customer_quarter,customer_period_month,customer_period_quarter
0,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,...,01/01/0001,01/01/0001,U106,Supplier1,1,2024,2,1,2024-02,2024Q1
1,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,...,01/01/0001,01/01/0001,U106,Supplier1,2,2024,2,1,2024-02,2024Q1
2,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,...,01/01/0001,01/01/0001,U106,Supplier1,3,2024,2,1,2024-02,2024Q1
3,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,...,01/01/0001,01/01/0001,U106,Supplier1,4,2024,2,1,2024-02,2024Q1
4,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,...,01/01/0001,01/01/0001,U106,Supplier1,5,2024,2,1,2024-02,2024Q1


,Date,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,Product_Category,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,...,SalesInvoiceLine_Quantity,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,supplier_iq_row_id,supplier_year,supplier_month,supplier_quarter,supplier_period_month,supplier_period_quarter
0,2024-02-26,0086,039-C-0692,38300,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,...,0,6.235348,14.21,4.61415752,1,2024,2,1,2024-02,2024Q1
1,2024-02-26,0086,039-C-0692,38322,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,...,0,8.130964,18.53,6.01691336,2,2024,2,1,2024-02,2024Q1
2,2024-02-26,0086,039-C-0692,38204,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,...,0,5.432344,12.38,4.01993456,3,2024,2,1,2024-02,2024Q1
3,2024-02-26,0086,039-C-0692,38492,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,...,0,11.351756,25.87,8.40029944,4,2024,2,1,2024-02,2024Q1
4,2024-02-26,0086,039-C-0692,38490,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,...,0,4.0589,9.25,3.003586,5,2024,2,1,2024-02,2024Q1


Created standardised dates and extra columns with date breakdowns from the single support file by month, year or quarter.


In [8]:
customer_amount_candidates = [
    "TotalClaimAmount",
    "UnitClaimAmount",
    "SalesDeliveryNote_NetAmountLessDiscountBase"
]

supplier_amount_candidates = [
    "TotalClaimAmount",
    "UnitClaimAmount",
    "Contract_Amount",
    "Contract_Percentage"
]

customer_amount_col = first_existing_col(customer, customer_amount_candidates)
supplier_reference_amount_col = first_existing_col(supplier, supplier_amount_candidates)

print("Customer claim amount column:", customer_amount_col)
print("Supplier support/reference amount column:", supplier_reference_amount_col)


Customer claim amount column: TotalClaimAmount
Supplier support/reference amount column: TotalClaimAmount


Converting variables to correct formats (numeric)

In [9]:
if customer_amount_col is not None:
    customer["credit_due_value"] = pd.to_numeric(
        customer[customer_amount_col],
        errors="coerce"
    )
else:
    raise ValueError("No customer claim amount column found.")

if supplier_reference_amount_col is not None:
    supplier["supplier_reference_value"] = pd.to_numeric(
        supplier[supplier_reference_amount_col],
        errors="coerce"
    )
else:
    supplier["supplier_reference_value"] = np.nan

display(customer[["credit_due_value"]].describe())
display(supplier[["supplier_reference_value"]].describe())


,credit_due_value
count,73320.000000
mean,34.202201
std,169.969396
min,-1741.740000
25%,1.448700
50%,3.642000
75%,11.628000
max,18500.000000


,supplier_reference_value
count,70559.000000
mean,32.527389
std,169.538267
min,-1741.740000
25%,1.411200
50%,3.540000
75%,10.929950
max,18500.000000


In [10]:
if "SalesDeliveryNoteLine_Quantity" in customer.columns:
    customer["customer_quantity"] = pd.to_numeric(
        customer["SalesDeliveryNoteLine_Quantity"], 
        errors="coerce"
    )

if "SalesInvoiceLine_Quantity" in customer.columns:
    customer["invoice_quantity"] = pd.to_numeric(
        customer["SalesInvoiceLine_Quantity"], 
        errors="coerce"
    )

if "Contract_MinimumQuantity" in supplier.columns:
    supplier["contract_minimum_quantity"] = pd.to_numeric(
        supplier["Contract_MinimumQuantity"], 
        errors="coerce"
    )


In [11]:
if "Product_CSQL_InvoiceCostForbranch" in customer.columns:
    customer["invoice_cost"] = pd.to_numeric(
        customer["Product_CSQL_InvoiceCostForbranch"], 
        errors="coerce"
    )

if "Product_CALC_FixedCostForBranch" in customer.columns:
    customer["fixed_cost"] = pd.to_numeric(
        customer["Product_CALC_FixedCostForBranch"], 
        errors="coerce"
    )

if "Product_CSQL_ListPriceForBranch" in customer.columns:
    customer["list_price"] = pd.to_numeric(
        customer["Product_CSQL_ListPriceForBranch"], 
        errors="coerce"
    )

if {"list_price", "invoice_cost"}.issubset(customer.columns):
    customer["estimated_gross_margin_before_credit"] = (
        customer["list_price"] - customer["invoice_cost"]
    )

if {"credit_due_value", "estimated_gross_margin_before_credit"}.issubset(customer.columns):
    customer["estimated_margin_if_credit_paid"] = (
        customer["estimated_gross_margin_before_credit"] + customer["credit_due_value"]
    )

    customer["estimated_margin_if_credit_not_paid"] = (
        customer["estimated_gross_margin_before_credit"]
    )

    customer["margin_at_risk_from_unpaid_credit"] = customer["credit_due_value"]

    customer["margin_risk_flag"] = np.where(
        customer["estimated_margin_if_credit_not_paid"] < 0,
        1,
        0
    )


In [12]:
possible_merge_keys = [
    "Contract_Number",
    "Contract_D_ContractNumber",
    "Product_ManufacturerProductCode"
]

merge_keys = [
    col for col in possible_merge_keys
    if col in customer.columns and col in supplier.columns
]

print("Merge keys being used:")
print(merge_keys)

if not merge_keys:
    raise ValueError("No merge keys found. Check cleaned column names.")

for col in merge_keys:
    customer[col] = normalise_key(customer[col])
    supplier[col] = normalise_key(supplier[col])


Merge keys being used:
['Contract_Number', 'Contract_D_ContractNumber', 'Product_ManufacturerProductCode']


The customer claim view and supplier support view are both derived from `SupportData.xlsx`. They are still matched on contract number, contract support ID and manufacturer product code so the analysis keeps a clear separation between claim-line grain and supplier-support evidence

Duplicates are not always negative in this scenario and can offer genuine business information so they are identified here rather than removed


In [13]:
customer_for_merge = customer.copy()
supplier_for_merge = supplier.copy()

claim_line_id_cols = [
    "customer_iq_row_id",
    "Date",
    "Customer",
    "Customer_Name",
    "Product",
    "Product_ManufacturerProductCode",
    "Contract_Number",
    "Contract_D_ContractNumber",
    "SalesInvoice_Number",
    "SalesDeliveryNote_Number",
    "SalesOrder_Number",
    "SalesCreditNote_Number",
    "credit_due_value"
]
customer_for_merge["claim_line_id"] = stable_line_id(customer_for_merge, claim_line_id_cols)

duplicate_grain_cols = [
    col for col in [
        "Customer",
        "Product",
        "Product_ManufacturerProductCode",
        "Contract_Number",
        "Contract_D_ContractNumber",
        "Date",
        "credit_due_value"
    ]
    if col in customer_for_merge.columns
]

if duplicate_grain_cols:
    customer_for_merge["claim_line_sequence_within_key"] = (
        customer_for_merge
        .groupby(duplicate_grain_cols, dropna=False)
        .cumcount()
        .add(1)
    )
else:
    customer_for_merge["claim_line_sequence_within_key"] = customer_for_merge.index + 1

customer_for_merge = customer_for_merge.rename(columns={"Date": "customer_date"})
supplier_for_merge = supplier_for_merge.rename(columns={"Date": "supplier_date"})

customer_for_merge["customer_date"] = pd.to_datetime(customer_for_merge["customer_date"], errors="coerce")
supplier_for_merge["supplier_date"] = pd.to_datetime(supplier_for_merge["supplier_date"], errors="coerce")

for col in merge_keys:
    customer_for_merge[col] = normalise_key(customer_for_merge[col])
    supplier_for_merge[col] = normalise_key(supplier_for_merge[col])

for col in merge_keys:
    print(f"\nMerge key check: {col}")
    customer_examples = (
        customer_for_merge[col]
        .dropna()
        .astype(str)
        .loc[lambda s: s.str.match(r"^0+\d+$", na=False)]
        .head(10)
        .tolist()
    )
    supplier_examples = (
        supplier_for_merge[col]
        .dropna()
        .astype(str)
        .loc[lambda s: s.str.match(r"^0+\d+$", na=False)]
        .head(10)
        .tolist()
    )
    print("Customer examples with leading zeroes:", customer_examples)
    print("Supplier examples with leading zeroes:", supplier_examples)

duplicate_key_summary_customer = (
    customer_for_merge
    .groupby(merge_keys, dropna=False)
    .size()
    .reset_index(name="customer_row_count")
    .sort_values("customer_row_count", ascending=False)
)

duplicate_key_summary_supplier = (
    supplier_for_merge
    .groupby(merge_keys, dropna=False)
    .size()
    .reset_index(name="supplier_row_count")
    .sort_values("supplier_row_count", ascending=False)
)

print("Customer duplicate key groups:")
display(duplicate_key_summary_customer.head(20))
print("Supplier duplicate key groups:")
display(duplicate_key_summary_supplier.head(20))

customer_for_merge = customer_for_merge.reset_index(names="customer_row_id")
supplier_before = len(supplier_for_merge)
supplier_for_merge = supplier_for_merge.sort_values(merge_keys + ["supplier_date"])
supplier_after = len(supplier_for_merge)

print("Supplier rows before sorting:", supplier_before)
print("Supplier rows after sorting:", supplier_after)
print("Supplier rows removed: 0")

customer_for_merge = customer_for_merge.sort_values("customer_date")
supplier_for_merge = supplier_for_merge.sort_values("supplier_date")

matched = pd.merge_asof(
    customer_for_merge,
    supplier_for_merge,
    left_on="customer_date",
    right_on="supplier_date",
    by=merge_keys,
    direction="nearest",
    tolerance=pd.Timedelta(days=180),
    suffixes=("_customer", "_supplier")
)

matched = matched.sort_values("customer_row_id").drop(columns=["customer_row_id"])
matched["_merge"] = np.where(matched["supplier_date"].notna(), "both", "left_only")
matched["days_between_customer_and_supplier"] = (matched["supplier_date"] - matched["customer_date"]).dt.days
matched["supplier_reference_match_status"] = np.where(
    matched["_merge"].eq("both"),
    "Supplier reference matched",
    "No supplier reference matched"
)

print("Customer rows before merge:", len(customer))
print("Matched rows after merge:", len(matched))
print("Distinct claim_line_id values:", matched["claim_line_id"].nunique())

if len(matched) != len(customer):
    raise ValueError(f"Merge changed row count. Customer rows: {len(customer)}, matched rows: {len(matched)}")

print("Row count check passed: one matched row per original customer row.")
display(matched["_merge"].value_counts(dropna=False))
display(matched["days_between_customer_and_supplier"].describe())



Merge key check: Contract_Number
Customer examples with leading zeroes: ['0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086']
Supplier examples with leading zeroes: ['0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086', '0086']

Merge key check: Contract_D_ContractNumber
Customer examples with leading zeroes: ['00018875', '00018875', '00018945', '00018945', '00018945', '00018945', '00018945', '00018945', '00023542', '00018945']
Supplier examples with leading zeroes: ['00018875', '00018875', '00018945', '00018945', '00018945', '00018945', '00018945', '00018945', '00023542', '00018945']

Merge key check: Product_ManufacturerProductCode
Customer examples with leading zeroes: ['0010020390', '0010021222', '0010021222', '0010020389', '0010021837', '0010021838', '0010021837', '0010020389', '0010021837', '0010021837']
Supplier examples with leading zeroes: ['0010020390', '0010021222', '0010021222', '0010020389', '0010021837', '0010021838', '0010021

,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,customer_row_count
7615,0843,6114466,228335,1129
7556,0346,00995972/R7,087N6609,1048
4437,0000860,NIHE PRICING,V222P,934
7560,0349,00995972/R7,087N6579,872
6478,0126,083-C-0443,38280,473
9234,1871,117028080,7733600373,470
93,0000009,083-C-0373,262001,354
7573,0362,IRECS00216,CR2P-222301,312
6509,0126,083-C-0443,38450,240
6430,0126,083-C-0443,38010,234


Supplier duplicate key groups:


,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,supplier_row_count
7556,0346,00995972/R7,087N6609,917
7615,0843,6114466,228335,778
7560,0349,00995972/R7,087N6579,675
4437,0000860,NIHE PRICING,V222P,657
6478,0126,083-C-0443,38280,458
9234,1871,117028080,7733600373,418
93,0000009,083-C-0373,262001,327
7573,0362,IRECS00216,CR2P-222301,267
6509,0126,083-C-0443,38450,235
6430,0126,083-C-0443,38010,232


Supplier rows before sorting: 70559
Supplier rows after sorting: 70559
Supplier rows removed: 0
Customer rows before merge: 73320
Matched rows after merge: 73320
Distinct claim_line_id values: 73320
Row count check passed: one matched row per original customer row.


_merge
both    73320
Name: count, dtype: int64

count    73320.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: days_between_customer_and_supplier, dtype: float64

In [14]:
canonical_fields = {
    "Contract_Supplier_Name": ["Contract_Supplier_Name", "Contract_Supplier_Name_supplier", "Contract_Supplier_Name_customer"],
    "Contract_Supplier_Code": ["Contract_Supplier_Code", "Contract_Supplier_Code_supplier", "Contract_Supplier_Code_customer"],
    "Product_Category": ["Product_Category", "Product_Category_supplier", "Product_Category_customer"],
    "Contract_Description": ["Contract_Description", "Contract_Description_supplier", "Contract_Description_customer"],
    "Contract_Expression": ["Contract_Expression", "Contract_Expression_supplier", "Contract_Expression_customer"]
}

for target_col, candidate_cols in canonical_fields.items():
    available_cols = [col for col in candidate_cols if col in matched.columns]
    if available_cols:
        matched[target_col] = matched[available_cols].bfill(axis=1).iloc[:, 0]

matched["supplier_reference_match_status"] = np.select(
    [
        matched["_merge"].eq("both"),
        matched["_merge"].eq("left_only")
    ],
    [
        "Supplier reference matched",
        "No supplier reference matched"
    ],
    default="Check"
)

display(matched["supplier_reference_match_status"].value_counts(dropna=False))


C:\Users\cathalhe\AppData\Local\Temp\ipykernel_40572\2906223177.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matched[target_col] = matched[available_cols].bfill(axis=1).iloc[:, 0]
C:\Users\cathalhe\AppData\Local\Temp\ipykernel_40572\2906223177.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matched[target_col] = matched[available_cols].bfill(axis=1).iloc[:, 0]
C:\Users\cathalhe\AppData\Local\Temp\ipykernel_40572\2906223177.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a futu

supplier_reference_match_status
Supplier reference matched    73320
Name: count, dtype: int64

In [15]:
status_col = first_existing_col(matched, ["Status", "Status_customer", "claim_status"])

if status_col is not None:
    matched["iq_claim_status"] = (
        matched[status_col]
        .astype("string")
        .str.strip()
        .str.title()
    )
else:
    matched["iq_claim_status"] = "Unknown"

matched["is_claimed_in_iq"] = matched["iq_claim_status"].str.contains("Claimed", case=False, na=False).astype(int)
matched["supplier_match_found"] = matched["supplier_reference_match_status"].eq("Supplier reference matched").astype(int)

credit_note_cols = first_existing_cols(matched, [
    "SalesCreditNote_Number",
    "SalesCreditNote_Number_customer",
    "CreditNote_Number",
    "Credit_Note_Number",
    "SalesReturnNote_Number",
    "SalesReturnNote_Number_customer"
])

if credit_note_cols:
    matched["has_credit_note_reference"] = (
        matched[credit_note_cols]
        .astype("string")
        .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA, "": pd.NA})
        .notna()
        .any(axis=1)
        .astype(int)
    )
else:
    matched["has_credit_note_reference"] = 0

credit_received_candidates = [
    "credit_received_value",
    "CreditReceivedValue",
    "Credit_Received_Value",
    "SupplierCreditReceivedValue",
    "Supplier_Credit_Received_Value",
    "AmountReceived",
    "Amount_Received",
    "PaidAmount",
    "Paid_Amount",
    "SupplierPaidAmount",
    "Supplier_Paid_Amount"
]

credit_received_source_col = first_existing_col(matched, credit_received_candidates)

if credit_received_source_col is not None:
    matched["credit_received_value"] = pd.to_numeric(matched[credit_received_source_col], errors="coerce").fillna(0)
    matched["credit_received_source"] = credit_received_source_col
else:
    matched["credit_received_value"] = 0.0
    matched["credit_received_source"] = "No explicit received-credit value in source data"

matched["recovery_evidence"] = np.select(
    [
        matched["credit_received_value"].gt(0),
        matched["has_credit_note_reference"].eq(1),
        matched["is_claimed_in_iq"].eq(1),
        matched["supplier_match_found"].eq(1)
    ],
    [
        "Explicit received-credit value",
        "Credit note/reference present - value confirmation required",
        "Claimed in IQ - receipt not confirmed",
        "Supplier support reference matched - receipt not confirmed"
    ],
    default="No recovery evidence"
)

display(pd.crosstab(matched["iq_claim_status"], matched["supplier_reference_match_status"], dropna=False))
display(matched["recovery_evidence"].value_counts(dropna=False))


supplier_reference_match_status,Supplier reference matched
iq_claim_status,
Claimed,73317
Unclaimed,3


recovery_evidence
Claimed in IQ - receipt not confirmed                          71572
Credit note/reference present - value confirmation required     1748
Name: count, dtype: int64

## Credit Recovery Logic

This cell separates support owed from credit actually received


In [16]:
matched["claim_date"] = matched["customer_date"]

matched["days_outstanding"] = (
    pd.Timestamp.today().normalize() - matched["claim_date"]
).dt.days

matched["ageing_band"] = pd.cut(
    matched["days_outstanding"],
    bins=[-np.inf, 30, 60, 90, np.inf],
    labels=["0-30", "31-60", "61-90", "90+"]
)

display(
    matched[[
        "customer_date",
        "supplier_date",
        "days_between_customer_and_supplier",
        "claim_date",
        "days_outstanding",
        "ageing_band"
    ]].head(10)
)


,customer_date,supplier_date,days_between_customer_and_supplier,claim_date,days_outstanding,ageing_band
0,2024-02-26,2024-02-26,0,2024-02-26,928,90+
67,2024-02-26,2024-02-26,0,2024-02-26,928,90+
66,2024-02-26,2024-02-26,0,2024-02-26,928,90+
65,2024-02-26,2024-02-26,0,2024-02-26,928,90+
70,2024-02-26,2024-02-26,0,2024-02-26,928,90+
63,2024-02-26,2024-02-26,0,2024-02-26,928,90+
62,2024-02-26,2024-02-26,0,2024-02-26,928,90+
61,2024-02-26,2024-02-26,0,2024-02-26,928,90+
60,2024-02-26,2024-02-26,0,2024-02-26,928,90+
59,2024-02-26,2024-02-26,0,2024-02-26,928,90+


In [17]:
matched["credit_due_value"] = pd.to_numeric(matched["credit_due_value"], errors="coerce").fillna(0)
matched["credit_received_value"] = pd.to_numeric(matched["credit_received_value"], errors="coerce").fillna(0)

matched["outstanding_credit_value"] = (
    matched["credit_due_value"] - matched["credit_received_value"]
).clip(lower=0)

matched["credit_recovery_status"] = np.select(
    [
        matched["credit_due_value"].le(0),
        matched["credit_received_value"].ge(matched["credit_due_value"]) & matched["credit_due_value"].gt(0),
        matched["credit_received_value"].gt(0) & matched["outstanding_credit_value"].gt(0),
        matched["supplier_match_found"].eq(1) & matched["is_claimed_in_iq"].eq(1),
        matched["supplier_match_found"].eq(1),
        matched["is_claimed_in_iq"].eq(1),
        matched["has_credit_note_reference"].eq(1)
    ],
    [
        "No positive credit due",
        "Recovered",
        "Partially recovered",
        "Supplier support matched and claimed - receipt not confirmed",
        "Supplier support matched - receipt not confirmed",
        "Claimed in IQ - awaiting supplier credit",
        "Credit note/reference present - value confirmation required"
    ],
    default="Unclaimed or no recovery evidence"
)

matched["is_credit_owed_to_us"] = np.where(matched["outstanding_credit_value"] > 0, 1, 0)
matched["recovered_credit_rate"] = pd.Series(
    np.where(
        matched["credit_due_value"].gt(0),
        matched["credit_received_value"] / matched["credit_due_value"],
        np.nan
    ),
    index=matched.index).clip(upper=1)

display(matched["credit_recovery_status"].value_counts(dropna=False))
display(matched[[
    "credit_due_value",
    "credit_received_value",
    "outstanding_credit_value",
    "supplier_reference_match_status",
    "credit_recovery_status",
    "recovery_evidence"]].head(10))


credit_recovery_status
Supplier support matched and claimed - receipt not confirmed    71444
No positive credit due                                           1876
Name: count, dtype: int64

,credit_due_value,credit_received_value,outstanding_credit_value,supplier_reference_match_status,credit_recovery_status,recovery_evidence
0,8.2012,0.0,8.2012,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
67,3.5663,0.0,3.5663,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
66,2.3819,0.0,2.3819,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
65,9.9559,0.0,9.9559,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
70,5.3428,0.0,5.3428,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
63,7.0059,0.0,7.0059,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
62,3.8220,0.0,3.8220,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
61,3.7477,0.0,3.7477,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
60,3.8680,0.0,3.8680,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed
59,1.8094,0.0,1.8094,Supplier reference matched,Supplier support matched and claimed - receipt...,Claimed in IQ - receipt not confirmed


In [18]:
if "credit_due_value" in matched.columns:
    matched["claim_value_band"] = pd.cut(
        matched["credit_due_value"],
        bins=[-np.inf, 0, 50, 250, 1000, np.inf],
        labels=["Zero or negative", "0-50", "50-250", "250-1000", "1000+"]
    )


Flags for supplier payment status

In [19]:
matched["high_value_review"] = np.where(
    matched["outstanding_credit_value"].fillna(0).abs() >= 250,
    1,
    0
)

matched["aged_review"] = np.where(
    matched["days_outstanding"].fillna(0) > 60,
    1,
    0
)

matched["margin_risk_review"] = np.where(
    matched.get(
        "margin_at_risk_from_unpaid_credit",
        pd.Series(0, index=matched.index)
    ).fillna(0).abs() >= 100,
    1,
    0
)

matched["requires_review"] = np.where(
    (matched["is_credit_owed_to_us"].eq(1))
    | (matched["high_value_review"].eq(1))
    | (matched["aged_review"].eq(1))
    | (matched["margin_risk_review"].eq(1)),
    1,
    0
)

matched["review_reason"] = np.select(
    [
        matched["is_credit_owed_to_us"].eq(1) & matched["aged_review"].eq(1),
        matched["is_credit_owed_to_us"].eq(1),
        matched["high_value_review"].eq(1),
        matched["margin_risk_review"].eq(1)
    ],
    [
        "Outstanding credit owed and aged over 60 days",
        "Outstanding credit owed by supplier",
        "High value credit requiring review",
        "Unpaid credit creates margin risk"
    ],
    default="No immediate review"
)


In [20]:
for name in [
    "matched", "report_tables", "df_model_base", "quality_report", "missing_after_cleaning", "gl_account_summary",
    "gl_account_summary_clean", "supplier_follow_up_priority_list", "model_results",
    "classification_results", "regression_results", "valid_date_check", "summary",
    "year_view", "month_view", "supplier_summary", "monthly_summary"
]:
    if name in globals():
        obj = globals()[name]
        shape = getattr(obj, "shape", "")
        print(name, shape)
        try:
            display(obj.head(5))
        except AttributeError:
            display(obj)


matched (73320, 110)


,customer_date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,outstanding_credit_value,credit_recovery_status,is_credit_owed_to_us,recovered_credit_rate,claim_value_band,high_value_review,aged_review,margin_risk_review,requires_review,review_reason
0,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,...,8.2012,Supplier support matched and claimed - receipt...,1,0.0,0-50,0,1,0,1,Outstanding credit owed and aged over 60 days
67,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,...,3.5663,Supplier support matched and claimed - receipt...,1,0.0,0-50,0,1,0,1,Outstanding credit owed and aged over 60 days
66,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,...,2.3819,Supplier support matched and claimed - receipt...,1,0.0,0-50,0,1,0,1,Outstanding credit owed and aged over 60 days
65,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,...,9.9559,Supplier support matched and claimed - receipt...,1,0.0,0-50,0,1,0,1,Outstanding credit owed and aged over 60 days
70,2024-02-26,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,...,5.3428,Supplier support matched and claimed - receipt...,1,0.0,0-50,0,1,0,1,Outstanding credit owed and aged over 60 days


gl_account_summary (21, 14)


,Code,Description,D_MerlinAccountCode,CurrentBalance,gl_account_code,gl_account_description,gl_supplier_key,gl_current_balance,gl_current_balance_abs,gl_entry_rows,gl_first_entry_date,gl_last_entry_date,gl_total_credit_amount,gl_latest_running_balance
0,01-85001,Supplier13 E071 Claimbacks,NaN,-242784.06,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,-242784.06,242784.06,105,2026-01-05,2026-03-07,242784.06,149850.93
1,01-85002,Supplier2 E005 Claimbacks,NaN,-6240,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,-6240.00,6240.00,1,NaT,NaT,6240.00,6240.00
2,01-85003,Supplier2GB E005GB Claimbacks,NaN,0,01-85003,Supplier2GB E005GB Claimbacks,SUPPLIER2GB,0.00,0.00,0,NaT,NaT,NaN,NaN
3,01-85007,Supplier9 E022 Claimbacks,NaN,-5774.89,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,-5774.89,5774.89,5,2026-04-06,2026-10-07,5774.89,4829.89
4,01-85017,Supplier3 E055 Claimbacks,NaN,-5550,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,-5550.00,5550.00,4,2026-10-04,2026-10-04,5550.00,5550.00
